# 05 — Financial Analysis (RQ3)

**Research Question:** Where is the system losing money?

SUS reimburses hospitals per admission. Longer stays cost more bed-days but don't
necessarily produce better outcomes. We quantify the financial cost of inefficiency.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from shared import (
    load_kidney, load_hospital_tags, setup_plot_style,
    save_plot, save_metrics, RAW_SIA_DIR, PROC_NAMES,
)

setup_plot_style()
kidney = load_kidney()
tags = load_hospital_tags()
print(f"Loaded {len(kidney):,} admissions, {len(tags)} hospitals")

Loaded 206,500 admissions, 510 hospitals


## 1. Cost by Procedure Category

In [2]:
cost_by_cat = kidney.groupby("proc_category").agg(
    total_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="sum"),
    mean_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="mean"),
    median_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="median"),
    count=pd.NamedAgg(column="VAL_TOT", aggfunc="count"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
).sort_values("total_cost", ascending=False)

cost_by_cat["pct_total_cost"] = cost_by_cat["total_cost"] / cost_by_cat["total_cost"].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

(cost_by_cat["total_cost"] / 1e6).plot.barh(ax=axes[0], color="#2563EB")
axes[0].set_title("Total Cost by Category")
axes[0].set_xlabel("BRL (millions)")
axes[0].invert_yaxis()

cost_by_cat["mean_cost"].plot.barh(ax=axes[1], color="#059669")
axes[1].set_title("Mean Cost per Admission")
axes[1].set_xlabel("BRL")
axes[1].invert_yaxis()

fig.suptitle("Financial Breakdown by Procedure Category", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "cost_by_category", prefix="05")
print(cost_by_cat.to_string())

  Saved: 05_cost_by_category.png
                 total_cost    mean_cost  median_cost  count  mean_los  pct_total_cost
proc_category                                                                         
SURGICAL        87186140.37   983.143406       831.46  88681  2.614777       46.419271
CLINICAL_MGMT   35107578.59  1508.381465      1234.41  23275  2.388786       18.691827
SURGICAL_MGMT   25614469.73  1243.601968      1120.11  20597  2.247852       13.637546
INTERVENTIONAL  19640671.06   976.516236       946.11  20113  2.129916       10.457002
DIAGNOSTIC      15298063.81   368.743554       280.41  41487  2.687734        8.144930
OTHER            3789535.66  1073.827050       539.88   3529  4.030037        2.017608
OBSERVATION      1186696.33   134.576585        73.53   8818  0.580517        0.631816


## 2. Excess Cost from Long Stays

In [3]:
cat_stats = kidney.groupby("proc_category")["DIAS_PERM"].agg(
    median="median", p75=lambda x: x.quantile(0.75), mean="mean", count="count"
).sort_values("count", ascending=False)

print("=== LOS Benchmarks by Category ===")
print(cat_stats.to_string())
print()

kidney_cost = kidney.merge(cat_stats[["median", "p75"]], on="proc_category", how="left")

kidney_cost["cost_per_day"] = np.where(
    kidney_cost["DIAS_PERM"] > 0,
    kidney_cost["VAL_TOT"] / kidney_cost["DIAS_PERM"],
    0
)

# Scenario A: median benchmark (upper-bound)
kidney_cost["excess_days_median"] = (kidney_cost["DIAS_PERM"] - kidney_cost["median"]).clip(lower=0)
kidney_cost["excess_cost_median"] = kidney_cost["excess_days_median"] * kidney_cost["cost_per_day"]

# Scenario B: P75 benchmark (conservative — only flags the worst quartile)
kidney_cost["excess_days_p75"] = (kidney_cost["DIAS_PERM"] - kidney_cost["p75"]).clip(lower=0)
kidney_cost["excess_cost_p75"] = kidney_cost["excess_days_p75"] * kidney_cost["cost_per_day"]

total_cost = kidney_cost["VAL_TOT"].sum()

for label, days_col, cost_col in [
    ("Median (upper-bound)", "excess_days_median", "excess_cost_median"),
    ("P75 (conservative)", "excess_days_p75", "excess_cost_p75"),
]:
    ec = kidney_cost[cost_col].sum()
    ed = kidney_cost[days_col].sum()
    n_flagged = (kidney_cost[days_col] > 0).sum()
    print(f"\n--- Scenario: {label} ---")
    print(f"  Total system cost:  R${total_cost:>14,.0f}")
    print(f"  Excess cost:        R${ec:>14,.0f} ({ec/total_cost*100:.1f}%)")
    print(f"  Excess bed-days:    {ed:>14,.0f}")
    print(f"  Admissions flagged: {n_flagged:>14,} / {len(kidney_cost):,} ({n_flagged/len(kidney_cost)*100:.1f}%)")

excess_by_cat_median = kidney_cost.groupby("proc_category")["excess_cost_median"].sum().sort_values(ascending=False)
excess_by_cat_p75 = kidney_cost.groupby("proc_category")["excess_cost_p75"].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

kidney_cost["excess_days_median"].clip(upper=15).hist(
    bins=16, ax=axes[0], color="#DC2626", edgecolor="white", alpha=0.8)
axes[0].set_title("Excess Days (Median benchmark)")
axes[0].set_xlabel("Excess days (capped at 15)")

(excess_by_cat_median / 1e6).plot.barh(ax=axes[1], color="#DC2626")
axes[1].set_title("Excess Cost — Median Benchmark")
axes[1].set_xlabel("BRL (millions)")
axes[1].invert_yaxis()

(excess_by_cat_p75 / 1e6).plot.barh(ax=axes[2], color="#D97706")
axes[2].set_title("Excess Cost — P75 Benchmark (conservative)")
axes[2].set_xlabel("BRL (millions)")
axes[2].invert_yaxis()

fig.suptitle("Cost of Excess LOS — Two Scenarios", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "excess_cost", prefix="05")

print("\n⚠ Assumption: cost_per_day = VAL_TOT / DIAS_PERM (linear).")
print("  Real costs are front-loaded (surgery, anesthesia), so excess-day cost is likely LOWER.")
print("  These are upper-bound estimates.")

# Store for downstream use
total_excess_median = kidney_cost["excess_cost_median"].sum()
total_excess_p75 = kidney_cost["excess_cost_p75"].sum()
excess_pct_median = total_excess_median / total_cost * 100
excess_pct_p75 = total_excess_p75 / total_cost * 100

=== LOS Benchmarks by Category ===
                median  p75      mean  count
proc_category                               
SURGICAL           2.0  3.0  2.614777  88681
DIAGNOSTIC         2.0  3.0  2.687734  41487
CLINICAL_MGMT      2.0  3.0  2.388786  23275
SURGICAL_MGMT      2.0  3.0  2.247852  20597
INTERVENTIONAL     2.0  2.0  2.129916  20113
OBSERVATION        0.0  1.0  0.580517   8818
OTHER              3.0  5.0  4.030037   3529


--- Scenario: Median (upper-bound) ---
  Total system cost:  R$   187,823,156
  Excess cost:        R$    41,771,443 (22.2%)
  Excess bed-days:           212,761
  Admissions flagged:         67,355 / 206,500 (32.6%)

--- Scenario: P75 (conservative) ---
  Total system cost:  R$   187,823,156
  Excess cost:        R$    27,435,391 (14.6%)
  Excess bed-days:           149,423
  Admissions flagged:         40,035 / 206,500 (19.4%)


  Saved: 05_excess_cost.png

⚠ Assumption: cost_per_day = VAL_TOT / DIAS_PERM (linear).
  Real costs are front-loaded (surgery, anesthesia), so excess-day cost is likely LOWER.
  These are upper-bound estimates.


## 3. SIH vs SIA — The Admission Premium

In [4]:
sia_files = sorted(RAW_SIA_DIR.glob("PASP*.parquet"))
premium = pd.DataFrame()

if sia_files:
    sih_costs = kidney.groupby("PROC_REA").agg(
        sih_mean_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="mean"),
        sih_count=pd.NamedAgg(column="VAL_TOT", aggfunc="count"),
    )
    sih_costs.index = sih_costs.index.astype(str).str.strip()

    sia_sample = sia_files[-12:]
    sia_frames = []
    for f in sia_sample:
        try:
            df = pd.read_parquet(f, columns=["PA_PROC_ID", "PA_VALAPR"])
            df["PA_PROC_ID"] = df["PA_PROC_ID"].astype(str).str.strip()
            matched = df[df["PA_PROC_ID"].isin(sih_costs.index)]
            if not matched.empty:
                sia_frames.append(matched)
        except Exception:
            pass

    if sia_frames:
        sia_data = pd.concat(sia_frames, ignore_index=True)
        sia_data["PA_VALAPR"] = pd.to_numeric(sia_data["PA_VALAPR"], errors="coerce")
        sia_costs = sia_data.groupby("PA_PROC_ID").agg(
            sia_mean_cost=pd.NamedAgg(column="PA_VALAPR", aggfunc="mean"),
            sia_count=pd.NamedAgg(column="PA_VALAPR", aggfunc="count"),
        )

        premium = sih_costs.join(sia_costs, how="inner")
        premium["admission_premium"] = premium["sih_mean_cost"] / premium["sia_mean_cost"].replace(0, np.nan)
        premium["proc_name"] = premium.index.map(PROC_NAMES)
        premium = premium[np.isfinite(premium["admission_premium"])]
        premium = premium.sort_values("admission_premium", ascending=False)

        print(f"Procedures found in both SIH and SIA: {len(premium)}")
        print(premium[["proc_name", "sih_mean_cost", "sia_mean_cost", "admission_premium",
                       "sih_count", "sia_count"]].to_string())

        if len(premium) > 0:
            fig, ax = plt.subplots(figsize=(12, max(4, len(premium) * 0.6)))
            labels = [PROC_NAMES.get(idx, idx) for idx in premium.index]
            ax.barh(range(len(premium)), premium["admission_premium"], color="#D97706")
            ax.set_yticks(range(len(premium)))
            ax.set_yticklabels(labels, fontsize=9)
            ax.axvline(1, color="black", linestyle="--", linewidth=0.8)
            ax.set_title("Admission Premium: Inpatient / Outpatient Cost Ratio")
            ax.set_xlabel("Cost Multiplier (SIH / SIA)")
            ax.invert_yaxis()
            fig.tight_layout()
            save_plot(fig, "admission_premium", prefix="05")
    else:
        print("No matching procedures found in SIA.")
else:
    print("SIA data not available.")

Procedures found in both SIH and SIA: 13
                        proc_name  sih_mean_cost  sia_mean_cost  admission_premium  sih_count  sia_count
0415040035                    NaN     963.790000       0.550102        1752.020404          2        392
0409010090                    NaN     718.499130      32.680000          21.985898         23        285
0409010294     Kidney Exploration    2029.612383     105.336000          19.267984        554          5
0409040010                    NaN     211.060000      12.970000          16.272938          1         16
0409020079                    NaN     365.934286      32.680000          11.197500          7          8
0409040215                    NaN     256.970000      34.100000           7.535777          3         10
0409020176                    NaN     382.233824      61.380000           6.227335         34          5
0409010170  Open Ureterolithotomy     734.879795     131.707317           5.579643      40973        123
0406020620    

## 4. Cost Variation Across Hospitals

In [5]:
hosp_costs = kidney.groupby("CNES").agg(
    n=pd.NamedAgg(column="CNES", aggfunc="count"),
    mean_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="mean"),
    total_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="sum"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
).reset_index()
hosp_costs = hosp_costs[hosp_costs["n"] >= 20]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(hosp_costs["mean_los"], hosp_costs["mean_cost"],
               s=hosp_costs["n"] / 5, alpha=0.6, color="#2563EB")
axes[0].set_title("Hospital Mean LOS vs Mean Cost")
axes[0].set_xlabel("Mean LOS (days)")
axes[0].set_ylabel("Mean cost (BRL)")

cpd = kidney[kidney["DIAS_PERM"] > 0].copy()
cpd["cost_per_day"] = cpd["VAL_TOT"] / cpd["DIAS_PERM"]
cpd["cost_per_day"].clip(upper=2000).hist(bins=50, ax=axes[1], color="#059669",
                                           edgecolor="white", alpha=0.8)
axes[1].set_title("Cost per Bed-Day Distribution")
axes[1].set_xlabel("BRL per day (capped at 2000)")
axes[1].set_ylabel("Count")

fig.suptitle("Hospital Cost Variation", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "hospital_cost_variation", prefix="05")

print(f"\nHospitals with ≥20 admissions: {len(hosp_costs)}")
print(f"Mean cost range: R${hosp_costs['mean_cost'].min():.0f} — R${hosp_costs['mean_cost'].max():.0f}")
print(f"Cost IQR: R${hosp_costs['mean_cost'].quantile(0.25):.0f} — R${hosp_costs['mean_cost'].quantile(0.75):.0f}")

  Saved: 05_hospital_cost_variation.png

Hospitals with ≥20 admissions: 313
Mean cost range: R$118 — R$1758
Cost IQR: R$288 — R$874


## 5. Deep Dive — Hospital Financial Efficiency

Cost deviation from median is NOT efficiency. A cheap hospital that kills patients is not efficient.

**Efficiency = outcome per unit of cost.** For each hospital, within each procedure, we measure:
- Did the patient survive? (resolution)
- How long did it take? (LOS vs procedure median)
- How much did it cost? (cost vs procedure median)

**Approach:** For each procedure × hospital combination (min 10 cases), compute:
1. `cost_ratio` = hospital median cost / system median cost (1.0 = average)
2. `los_ratio` = hospital median LOS / system median LOS (1.0 = average)
3. `mortality_rate` = deaths / admissions
4. `resolution_rate` = (alive + LOS ≤ procedure median) / admissions

Then aggregate across procedures, weighted by volume.

In [6]:
from scipy import stats
from shared import load_cnes_names, hospital_name

names_df = load_cnes_names()

# --- System-wide baselines per procedure ---
proc_baselines = kidney.groupby("PROC_REA").agg(
    sys_n=("VAL_TOT", "count"),
    sys_median_cost=("VAL_TOT", "median"),
    sys_median_los=("DIAS_PERM", "median"),
    sys_mortality=("MORTE", "mean"),
).reset_index()
proc_baselines = proc_baselines[proc_baselines["sys_n"] >= 50]

print(f"Procedures with ≥50 cases (for stable baselines): {len(proc_baselines)}")
print(f"\n{'Procedure':35s} {'N':>7s} {'Med Cost':>10s} {'Med LOS':>8s} {'Mort%':>7s}")
print("-" * 70)
from shared import PROC_NAMES
for _, p in proc_baselines.sort_values("sys_n", ascending=False).iterrows():
    name = PROC_NAMES.get(p["PROC_REA"], p["PROC_REA"])
    print(f"  {name[:33]:33s} {p['sys_n']:>7,.0f} R${p['sys_median_cost']:>8,.0f} {p['sys_median_los']:>7.1f}d {p['sys_mortality']*100:>6.2f}%")

# --- Per hospital × procedure: compute efficiency metrics ---
MIN_HOSP_PROC = 10
hp = kidney[kidney["PROC_REA"].isin(proc_baselines["PROC_REA"])].copy()
hp = hp.merge(proc_baselines, on="PROC_REA", how="left")

# Pre-compute resolution flag per admission
hp["resolved"] = ((hp["MORTE"] == 0) & (hp["DIAS_PERM"] <= hp["sys_median_los"])).astype(int)

hp_agg = hp.groupby(["CNES", "PROC_REA"]).agg(
    n=("VAL_TOT", "count"),
    hosp_median_cost=("VAL_TOT", "median"),
    hosp_median_los=("DIAS_PERM", "median"),
    hosp_mortality=("MORTE", "mean"),
    resolution_rate=("resolved", "mean"),
    sys_median_cost=("sys_median_cost", "first"),
    sys_median_los=("sys_median_los", "first"),
    sys_mortality=("sys_mortality", "first"),
).reset_index()
hp_agg = hp_agg[hp_agg["n"] >= MIN_HOSP_PROC]

hp_agg["cost_ratio"] = hp_agg["hosp_median_cost"] / hp_agg["sys_median_cost"]
hp_agg["los_ratio"] = hp_agg["hosp_median_los"] / hp_agg["sys_median_los"].replace(0, 0.5)

print(f"\nHospital × Procedure combinations with ≥{MIN_HOSP_PROC} cases: {len(hp_agg):,}")
print(f"Covering {hp_agg['n'].sum():,} admissions across {hp_agg['CNES'].nunique()} hospitals")

Procedures with ≥50 cases (for stable baselines): 32

Procedure                                 N   Med Cost  Med LOS   Mort%
----------------------------------------------------------------------
  Open Ureterolithotomy              40,973 R$     463     2.0d   0.78%
  Diagnostic Imaging (Urography)     40,657 R$     278     2.0d   0.18%
  Ureteroscopy (modern)              34,036 R$     954     1.0d   0.16%
  Clinical Management                23,275 R$   1,234     2.0d   0.33%
  Surgical Management                20,597 R$   1,120     2.0d   0.21%
  Ureteral Catheter                  13,145 R$     964     2.0d   0.21%
  Pyelolithotomy                      8,040 R$   1,180     3.0d   0.12%
  ER Observation                      5,851 R$      74     0.0d   0.03%
  Clinical Care (short)               2,967 R$      68     1.0d   0.24%
  ESWL Lithotripsy                    2,784 R$     403     0.0d   0.04%
  Percutaneous Nephrostomy            2,450 R$     550     1.0d   0.04%
  JJ Stent 

### 5a. Hospital Efficiency Score — Cost × Outcome

In [7]:
# --- Aggregate to hospital level: volume-weighted efficiency ---
MIN_N = 30

hosp_eff = hp_agg.groupby("CNES").apply(
    lambda g: pd.Series({
        "n": g["n"].sum(),
        "n_procs": len(g),
        "w_cost_ratio": np.average(g["cost_ratio"], weights=g["n"]),
        "w_los_ratio": np.average(g["los_ratio"], weights=g["n"]),
        "w_mortality": np.average(g["hosp_mortality"], weights=g["n"]),
        "w_resolution": np.average(g["resolution_rate"], weights=g["n"]),
    })
).reset_index()
hosp_eff = hosp_eff[hosp_eff["n"] >= MIN_N].copy()
hosp_eff["name"] = hosp_eff["CNES"].apply(lambda c: hospital_name(c, names_df))

# Efficiency score: high resolution at low cost
# resolution_rate ranges 0-1 (higher = better), cost_ratio ranges ~0.3-3 (lower = better)
# Score = resolution_rate / cost_ratio → higher = more efficient
hosp_eff["efficiency"] = hosp_eff["w_resolution"] / hosp_eff["w_cost_ratio"]

hosp_eff = hosp_eff.sort_values("efficiency", ascending=False)

print(f"Hospitals with ≥{MIN_N} admissions: {len(hosp_eff)}")
print(f"\nEfficiency = resolution_rate / cost_ratio")
print(f"  resolution_rate = % patients (survived AND LOS ≤ procedure median)")
print(f"  cost_ratio = hospital median cost / system median cost for same procedure")
print(f"  Higher = better (more resolution per unit of cost)\n")

# TOP 15
print("=" * 130)
print("TOP 15 MOST EFFICIENT (best outcome per unit of cost)")
print("=" * 130)
hdr = f"{'Hospital':40s} {'N':>6s} {'Eff':>6s} {'Resol%':>7s} {'Cost×':>6s} {'LOS×':>6s} {'Mort%':>6s} {'Procs':>5s}"
print(hdr)
print("-" * 130)
for _, r in hosp_eff.head(15).iterrows():
    print(f"{r['name'][:40]:40s} {r['n']:6.0f} {r['efficiency']:6.2f} {r['w_resolution']*100:6.1f}% {r['w_cost_ratio']:6.2f} {r['w_los_ratio']:6.2f} {r['w_mortality']*100:5.2f}% {r['n_procs']:5.0f}")

# BOTTOM 15
print()
print("=" * 130)
print("TOP 15 LEAST EFFICIENT (worst outcome per unit of cost)")
print("=" * 130)
print(hdr)
print("-" * 130)
for _, r in hosp_eff.tail(15).sort_values("efficiency").iterrows():
    print(f"{r['name'][:40]:40s} {r['n']:6.0f} {r['efficiency']:6.2f} {r['w_resolution']*100:6.1f}% {r['w_cost_ratio']:6.2f} {r['w_los_ratio']:6.2f} {r['w_mortality']*100:5.2f}% {r['n_procs']:5.0f}")

stat, pval = stats.mannwhitneyu(
    hosp_eff.head(15)["efficiency"], hosp_eff.tail(15)["efficiency"], alternative="greater")
print(f"\nMann-Whitney U (top vs bottom): U={stat:.0f}, p={pval:.2e}")

Hospitals with ≥30 admissions: 271

Efficiency = resolution_rate / cost_ratio
  resolution_rate = % patients (survived AND LOS ≤ procedure median)
  cost_ratio = hospital median cost / system median cost for same procedure
  Higher = better (more resolution per unit of cost)

TOP 15 MOST EFFICIENT (best outcome per unit of cost)
Hospital                                      N    Eff  Resol%  Cost×   LOS×  Mort% Procs
----------------------------------------------------------------------------------------------------------------------------------
HOSPITAL E MATERNIDADE REGIONAL REGENTE      58   1.47   93.1%   0.63   1.00  0.00%     1
COMPLEXO MUNICIPAL DE SAUDE                  36   1.40   88.9%   0.63   0.50  0.00%     1
UNIDADE DE RETAGUARDA DE URGENCIA E DIAG     48   1.37   93.8%   0.68   0.50  0.00%     1
HOSPITAL DR RENATO SILVA DE SOCORRO          88   1.31   83.0%   0.63   1.00  0.00%     1
HOSPITAL DE URGENCIA                        646   1.29   82.2%   0.63   0.57  0.00%     

/var/folders/8r/2hn86n416n58v77nhrr2_mhw0000gn/T/ipykernel_54338/2414680116.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hosp_eff = hp_agg.groupby("CNES").apply(


### 5b. Efficient vs Inefficient — Profile Comparison

In [8]:
top15 = hosp_eff.head(15)
bot15 = hosp_eff.tail(15)
top_cnes = set(top15["CNES"])
bot_cnes = set(bot15["CNES"])

top_adm = kidney[kidney["CNES"].isin(top_cnes)]
bot_adm = kidney[kidney["CNES"].isin(bot_cnes)]

metrics = [
    ("Resolution rate", "w_resolution", ".1%", True),
    ("Cost ratio (1=avg)", "w_cost_ratio", ".2f", True),
    ("LOS ratio (1=avg)", "w_los_ratio", ".2f", True),
    ("Mortality rate", "w_mortality", ".2%", True),
    ("Efficiency score", "efficiency", ".2f", True),
]

print(f"{'Metric':30s} {'Efficient (n=15)':>18s} {'Inefficient (n=15)':>20s}")
print("-" * 72)
for label, col, fmt, from_eff in metrics:
    t_val = top15[col].mean()
    b_val = bot15[col].mean()
    t_str = f"{t_val:{fmt}}"
    b_str = f"{b_val:{fmt}}"
    print(f"  {label:28s} {t_str:>18s} {b_str:>20s}")

print(f"\n--- Admission-level comparison (all patients from these hospitals) ---")
print(f"  Top 15: {len(top_adm):,} admissions | Bottom 15: {len(bot_adm):,} admissions\n")

adm_metrics = [
    ("Mean LOS (days)", lambda df: df["DIAS_PERM"].mean(), ".1f"),
    ("Mean Cost (R$)", lambda df: df["VAL_TOT"].mean(), ",.0f"),
    ("% Same-day (D0)", lambda df: (df["DIAS_PERM"] == 0).mean(), ".1%"),
    ("% Long stay (>7d)", lambda df: (df["DIAS_PERM"] > 7).mean(), ".1%"),
    ("Mortality", lambda df: df["MORTE"].mean(), ".2%"),
    ("% Emergency", lambda df: df["is_emergency"].mean(), ".1%"),
    ("% Diagnostic", lambda df: (df["proc_category"] == "DIAGNOSTIC").mean(), ".1%"),
    ("% Surgical", lambda df: (df["proc_category"] == "SURGICAL").mean(), ".1%"),
]
for label, fn, fmt in adm_metrics:
    t_val = fn(top_adm)
    b_val = fn(bot_adm)
    t_str = f"{t_val:{fmt}}"
    b_str = f"{b_val:{fmt}}"
    print(f"  {label:28s} {t_str:>18s} {b_str:>20s}")

Metric                           Efficient (n=15)   Inefficient (n=15)
------------------------------------------------------------------------
  Resolution rate                           85.6%                26.6%
  Cost ratio (1=avg)                         0.71                 1.48
  LOS ratio (1=avg)                          0.66                 1.82
  Mortality rate                            0.05%                0.13%
  Efficiency score                           1.22                 0.18

--- Admission-level comparison (all patients from these hospitals) ---
  Top 15: 2,849 admissions | Bottom 15: 3,138 admissions

  Mean LOS (days)                             1.1                  4.3
  Mean Cost (R$)                              466                  628
  % Same-day (D0)                           45.7%                 1.9%
  % Long stay (>7d)                          0.7%                13.3%
  Mortality                                 0.07%                0.32%
  % Emergency   

### 5c. Excess Cost Concentration — Which Hospitals Generate the Most Waste?

In [9]:
# --- Efficiency visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: cost ratio vs resolution rate
axes[0].scatter(hosp_eff["w_cost_ratio"], hosp_eff["w_resolution"] * 100,
               s=hosp_eff["n"] / 10, alpha=0.5, color="#6B7280")
for _, r in top15.iterrows():
    axes[0].scatter(r["w_cost_ratio"], r["w_resolution"] * 100,
                   s=80, color="#059669", edgecolor="black", linewidth=0.5, zorder=5)
for _, r in bot15.iterrows():
    axes[0].scatter(r["w_cost_ratio"], r["w_resolution"] * 100,
                   s=80, color="#DC2626", edgecolor="black", linewidth=0.5, zorder=5)
axes[0].axvline(1.0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
axes[0].set_xlabel("Cost Ratio (1.0 = system median)")
axes[0].set_ylabel("Resolution Rate (%)")
axes[0].set_title("Cost vs Outcome (green=top15, red=bottom15)")

# Bar: top 15 vs bottom 15 efficiency scores
eff_top = top15[["name", "efficiency"]].copy()
eff_bot = bot15[["name", "efficiency"]].copy()
combined = pd.concat([
    eff_top.assign(group="Top 15"),
    eff_bot.assign(group="Bottom 15")
])
combined["label"] = combined["name"].str[:25]
combined = combined.sort_values("efficiency", ascending=True)

colors = ["#059669" if g == "Top 15" else "#DC2626" for g in combined["group"]]
axes[1].barh(range(len(combined)), combined["efficiency"], color=colors)
axes[1].set_yticks(range(len(combined)))
axes[1].set_yticklabels(combined["label"], fontsize=7)
axes[1].set_xlabel("Efficiency Score (resolution / cost)")
axes[1].set_title("Top 15 vs Bottom 15 Efficiency")

fig.suptitle("Hospital Efficiency: Outcome per Unit of Cost", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "hospital_efficiency", prefix="05")

  Saved: 05_hospital_efficiency.png


### 5d. What Drives Efficiency Differences?

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Cost ratio vs efficiency
rho1, p1 = stats.spearmanr(hosp_eff["w_cost_ratio"], hosp_eff["efficiency"])
axes[0].scatter(hosp_eff["w_cost_ratio"], hosp_eff["efficiency"],
               s=hosp_eff["n"] / 10, alpha=0.5, color="#2563EB")
axes[0].set_title(f"Cost Ratio vs Efficiency (ρ={rho1:.2f})")
axes[0].set_xlabel("Cost Ratio (1=system median)")
axes[0].set_ylabel("Efficiency Score")

# 2. LOS ratio vs efficiency
rho2, p2 = stats.spearmanr(hosp_eff["w_los_ratio"], hosp_eff["efficiency"])
axes[1].scatter(hosp_eff["w_los_ratio"], hosp_eff["efficiency"],
               s=hosp_eff["n"] / 10, alpha=0.5, color="#7C3AED")
axes[1].set_title(f"LOS Ratio vs Efficiency (ρ={rho2:.2f})")
axes[1].set_xlabel("LOS Ratio (1=system median)")
axes[1].set_ylabel("Efficiency Score")

# 3. Volume vs efficiency
rho3, p3 = stats.spearmanr(hosp_eff["n"], hosp_eff["efficiency"])
axes[2].scatter(hosp_eff["n"], hosp_eff["efficiency"],
               s=30, alpha=0.5, color="#D97706")
axes[2].set_title(f"Volume vs Efficiency (ρ={rho3:.2f})")
axes[2].set_xlabel("Total admissions")
axes[2].set_ylabel("Efficiency Score")
axes[2].set_xscale("log")

fig.suptitle("What Drives Efficiency?", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "efficiency_drivers", prefix="05")

print(f"Correlations with efficiency score:")
print(f"  Cost ratio:  ρ={rho1:.3f} (p={p1:.2e})")
print(f"  LOS ratio:   ρ={rho2:.3f} (p={p2:.2e})")
print(f"  Volume:       ρ={rho3:.3f} (p={p3:.2e})")

  Saved: 05_efficiency_drivers.png
Correlations with efficiency score:
  Cost ratio:  ρ=-0.694 (p=2.88e-40)
  LOS ratio:   ρ=-0.596 (p=1.72e-27)
  Volume:       ρ=-0.203 (p=7.78e-04)


### 5e. Why Do Some Hospitals Have Low Resolution Rates?

The bottom 15 resolve only 26.6% of cases within the procedure median LOS.
Let's look at their procedure profile vs the top 15.

In [11]:
# Compare procedure-level metrics: bottom 15 vs top 15
top_hp = hp_agg[hp_agg["CNES"].isin(top_cnes)]
bot_hp = hp_agg[hp_agg["CNES"].isin(bot_cnes)]

print("=== PROCEDURE-LEVEL BREAKDOWN: TOP 15 vs BOTTOM 15 ===\n")
procs_in_both = set(top_hp["PROC_REA"]) & set(bot_hp["PROC_REA"])
print(f"Procedures present in both groups: {len(procs_in_both)}\n")

print(f"{'Procedure':35s} {'Top15 Res%':>10s} {'Bot15 Res%':>10s} {'Top15 Cost×':>12s} {'Bot15 Cost×':>12s}")
print("-" * 82)
for proc in sorted(procs_in_both):
    t = top_hp[top_hp["PROC_REA"] == proc]
    b = bot_hp[bot_hp["PROC_REA"] == proc]
    name = PROC_NAMES.get(proc, proc)
    t_res = np.average(t["resolution_rate"], weights=t["n"])
    b_res = np.average(b["resolution_rate"], weights=b["n"])
    t_cost = np.average(t["cost_ratio"], weights=t["n"])
    b_cost = np.average(b["cost_ratio"], weights=b["n"])
    print(f"  {name[:33]:33s} {t_res*100:9.1f}% {b_res*100:9.1f}% {t_cost:11.2f} {b_cost:11.2f}")

# What % of bottom 15 admissions are emergency?
print(f"\n=== ADMISSION PROFILE ===")
print(f"  Top 15: {top_adm['is_emergency'].mean()*100:.1f}% emergency, {(top_adm['proc_category']=='DIAGNOSTIC').mean()*100:.1f}% diagnostic")
print(f"  Bot 15: {bot_adm['is_emergency'].mean()*100:.1f}% emergency, {(bot_adm['proc_category']=='DIAGNOSTIC').mean()*100:.1f}% diagnostic")
print(f"\n  Top 15 proc mix: {top_adm.groupby('proc_category').size().sort_values(ascending=False).head(5).to_dict()}")
print(f"  Bot 15 proc mix: {bot_adm.groupby('proc_category').size().sort_values(ascending=False).head(5).to_dict()}")

=== PROCEDURE-LEVEL BREAKDOWN: TOP 15 vs BOTTOM 15 ===

Procedures present in both groups: 5

Procedure                           Top15 Res% Bot15 Res%  Top15 Cost×  Bot15 Cost×
----------------------------------------------------------------------------------
  Clinical Care (short)                  97.6%      96.4%        0.65        2.92
  Diagnostic Imaging (Urography)         79.1%      31.1%        0.64        1.51
  Open Ureterolithotomy                  98.3%      23.1%        0.86        1.44
  Ureteroscopy (modern)                  99.7%      12.3%        0.96        0.85
  Clinical Management                    95.2%      33.5%        0.82        0.86

=== ADMISSION PROFILE ===
  Top 15: 51.8% emergency, 40.6% diagnostic
  Bot 15: 90.6% emergency, 63.0% diagnostic

  Top 15 proc mix: {'DIAGNOSTIC': 1156, 'SURGICAL': 1127, 'CLINICAL_MGMT': 358, 'OBSERVATION': 148, 'OTHER': 45}
  Bot 15 proc mix: {'DIAGNOSTIC': 1977, 'SURGICAL': 489, 'OBSERVATION': 249, 'CLINICAL_MGMT': 221, '

## 5½. Desvio do Custo Mediano — Quem Cobra Mais pelo Mesmo Procedimento?

A eficiência (seção 5) mede resultado/custo. Mas também é relevante saber:
**quais hospitais recebem mais dinheiro pelo mesmo procedimento?**

Para cada internação, computamos: `resíduo = custo real − mediana do sistema para o mesmo PROC_REA`.
Um resíduo positivo = o hospital recebe mais pelo mesmo procedimento.

Isso **não é eficiência** — é desvio de preço. Um hospital pode ter resíduo alto por:
- Componentes adicionais na AIH (UTI, hemoderivados, reintervenção)
- Codificação mais agressiva
- Casos genuinamente mais graves dentro do mesmo código

In [12]:
# --- Cost residual per admission (procedure-level) ---
proc_median_cost = kidney.groupby("PROC_REA")["VAL_TOT"].median().rename("proc_median_cost")
kc = kidney.merge(proc_median_cost, on="PROC_REA", how="left")
kc["cost_residual"] = kc["VAL_TOT"] - kc["proc_median_cost"]

# Per-hospital aggregation
MIN_N_RESID = 30
hosp_resid = kc.groupby("CNES").agg(
    n=("CNES", "count"),
    mean_cost=("VAL_TOT", "mean"),
    mean_residual=("cost_residual", "mean"),
    total_residual=("cost_residual", "sum"),
    mean_los=("DIAS_PERM", "mean"),
    pct_emergency=("is_emergency", "mean"),
).reset_index()
hosp_resid = hosp_resid[hosp_resid["n"] >= MIN_N_RESID].copy()
hosp_resid["name"] = hosp_resid["CNES"].apply(lambda c: hospital_name(c, names_df))
hosp_resid = hosp_resid.sort_values("mean_residual")

def top_residual_driver(cnes, kc_df):
    h = kc_df[kc_df["CNES"] == cnes]
    proc_resid = h.groupby("proc_name")["cost_residual"].agg(["sum", "count", "mean"]).sort_values("sum")
    if len(proc_resid) == 0:
        return "—"
    if abs(proc_resid.iloc[0]["sum"]) >= abs(proc_resid.iloc[-1]["sum"]):
        row, name = proc_resid.iloc[0], proc_resid.index[0]
    else:
        row, name = proc_resid.iloc[-1], proc_resid.index[-1]
    return f"{name[:25]} (R${row['mean']:+.0f}/caso, n={row['count']:.0f})"

print(f"Hospitals with ≥{MIN_N_RESID} admissions: {len(hosp_resid)}")
print(f"Residual range: R${hosp_resid['mean_residual'].min():.0f} to R${hosp_resid['mean_residual'].max():.0f}\n")

# Top 15 lowest residual (cheapest for same procedure)
print("=" * 120)
print("TOP 15 — LOWEST COST DEVIATION (pay less than median for same procedure)")
print("=" * 120)
print(f"{'Hospital':40s} {'N':>5s} {'Cost':>8s} {'Resid':>8s} {'LOS':>5s}  {'Main Driver':50s}")
print("-" * 120)
for _, r in hosp_resid.head(15).iterrows():
    driver = top_residual_driver(r["CNES"], kc)
    print(f"{r['name'][:40]:40s} {r['n']:5.0f} R${r['mean_cost']:>6,.0f} R${r['mean_residual']:>+6,.0f} {r['mean_los']:5.1f}  {driver}")

print()
print("=" * 120)
print("TOP 15 — HIGHEST COST DEVIATION (pay more than median for same procedure)")
print("=" * 120)
print(f"{'Hospital':40s} {'N':>5s} {'Cost':>8s} {'Resid':>8s} {'LOS':>5s}  {'Main Driver':50s}")
print("-" * 120)
for _, r in hosp_resid.tail(15).sort_values("mean_residual", ascending=False).iterrows():
    driver = top_residual_driver(r["CNES"], kc)
    print(f"{r['name'][:40]:40s} {r['n']:5.0f} R${r['mean_cost']:>6,.0f} R${r['mean_residual']:>+6,.0f} {r['mean_los']:5.1f}  {driver}")

Hospitals with ≥30 admissions: 283
Residual range: R$-193 to R$1112

TOP 15 — LOWEST COST DEVIATION (pay less than median for same procedure)
Hospital                                     N     Cost    Resid   LOS  Main Driver                                       
------------------------------------------------------------------------------------------------------------------------
HOSPITAL SAO DOMINGOS NA PROV DE DEUS NH   244 R$   548 R$  -193   1.6  Clinical Management (R$-456/caso, n=97)
AMHE MED SOROCABA                          178 R$   768 R$  -176   1.8  Ureteroscopy (modern) (R$-179/caso, n=176)
SANTA CASA SANTA RITA DO PASSA QUATRO       44 R$   579 R$  -140   0.8  Ureteroscopy (modern) (R$-186/caso, n=30)
SANTA CASA DE SANTA CRUZ DAS PALMEIRAS      33 R$   217 R$  -125   2.3  Diagnostic Imaging (Urogr (R$-96/caso, n=18)
BENEFICENCIA PORTUGUESA DE AMPARO          191 R$   316 R$  -113   2.1  Diagnostic Imaging (Urogr (R$-98/caso, n=109)
PRONTO SOCORRO DA VILA DIRCE          

SANTA CASA DE PENAPOLIS                    285 R$ 1,065 R$  -100   1.4  Clinical Management (R$-153/caso, n=255)
SANTA CASA DE SALTO GRANDE                  34 R$   178 R$  -100   2.0  Diagnostic Imaging (Urogr (R$-100/caso, n=34)


HOSPITAL E MATERNIDADE MUNICIPAL SANTA A    92 R$   185 R$  -100   3.0  Diagnostic Imaging (Urogr (R$-98/caso, n=86)
SANTA CASA DE SANTA ADELIA                 111 R$   181 R$   -99   3.0  Diagnostic Imaging (Urogr (R$-98/caso, n=110)
SANTA CASA DE MISERICORDIA DE TAMBAU        98 R$   694 R$   -97   1.3  Ureteroscopy (modern) (R$-190/caso, n=51)
HOSPITAL ANGATUBA                           39 R$   182 R$   -96   2.6  Diagnostic Imaging (Urogr (R$-96/caso, n=39)
SANTA CASA DE TAQUARITUBA                   32 R$   197 R$   -96   2.5  Diagnostic Imaging (Urogr (R$-95/caso, n=30)

TOP 15 — HIGHEST COST DEVIATION (pay more than median for same procedure)
Hospital                                     N     Cost    Resid   LOS  Main Driver                                       
------------------------------------------------------------------------------------------------------------------------
SANTA CASA DE CACONDE                       61 R$ 1,758 R$+1,112   0.9  Ureteral Catheter (R$+1851

SANTA CASA DE SAO CARLOS                  2689 R$ 1,242 R$  +482   1.6  Clinical Management (R$+889/caso, n=1140)
SANTA CASA HOSP DR ARISTOTELES OLIVEIRA    179 R$ 1,145 R$  +465   4.1  Open Ureterolithotomy (R$+1782/caso, n=19)
HOSPITAL REGIONAL DE PIRACICABA           4797 R$ 1,207 R$  +453   1.0  Ureteroscopy (modern) (R$+541/caso, n=3023)
SANTA CASA DE SAO JOSE CAMPOS              301 R$ 1,381 R$  +445   3.3  Surgical Management (R$+337/caso, n=204)
HOSPITAL DIA SANTO AMARO                  5538 R$ 1,370 R$  +443   1.3  Ureteroscopy (modern) (R$+462/caso, n=5220)
HOSPITAL SANTO AMARO                       167 R$   857 R$  +434   4.8  Open Ureterolithotomy (R$+2928/caso, n=11)


In [13]:
# --- Cross-reference: cost deviation vs efficiency ---
cross = hosp_resid[["CNES", "mean_residual", "total_residual"]].merge(
    hosp_eff[["CNES", "efficiency", "w_resolution", "w_cost_ratio"]], on="CNES", how="inner"
)

rho, pval = stats.spearmanr(cross["mean_residual"], cross["efficiency"])
print(f"Correlation: cost residual vs efficiency score: ρ={rho:.3f} (p={pval:.2e})")
print(f"  → {'Negative' if rho < 0 else 'Positive'}: hospitals that charge more tend to be {'less' if rho < 0 else 'more'} efficient\n")

# Scatter plot
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(cross["mean_residual"], cross["efficiency"],
               s=30, alpha=0.5, color="#2563EB")
ax.axvline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axhline(1.0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
ax.set_xlabel("Mean Cost Residual (R$) — desvio do custo mediano")
ax.set_ylabel("Efficiency Score — resolução/custo")
ax.set_title(f"Cost Deviation vs Efficiency (ρ={rho:.2f})")

# Label quadrants
ax.text(ax.get_xlim()[0] * 0.9, ax.get_ylim()[1] * 0.95, "Cheap + Efficient", fontsize=9,
       color="#059669", fontweight="bold", ha="left")
ax.text(ax.get_xlim()[1] * 0.9, ax.get_ylim()[1] * 0.95, "Expensive + Efficient", fontsize=9,
       color="#D97706", fontweight="bold", ha="right")
ax.text(ax.get_xlim()[0] * 0.9, ax.get_ylim()[0] * 1.1, "Cheap + Inefficient", fontsize=9,
       color="#6B7280", fontweight="bold", ha="left")
ax.text(ax.get_xlim()[1] * 0.9, ax.get_ylim()[0] * 1.1, "Expensive + Inefficient", fontsize=9,
       color="#DC2626", fontweight="bold", ha="right")

fig.tight_layout()
save_plot(fig, "cost_deviation_vs_efficiency", prefix="05")

# Count per quadrant
cheap_eff = ((cross["mean_residual"] < 0) & (cross["efficiency"] > cross["efficiency"].median())).sum()
cheap_ineff = ((cross["mean_residual"] < 0) & (cross["efficiency"] <= cross["efficiency"].median())).sum()
exp_eff = ((cross["mean_residual"] >= 0) & (cross["efficiency"] > cross["efficiency"].median())).sum()
exp_ineff = ((cross["mean_residual"] >= 0) & (cross["efficiency"] <= cross["efficiency"].median())).sum()
print(f"Quadrant distribution:")
print(f"  Cheap + Efficient:     {cheap_eff} ({cheap_eff/len(cross)*100:.0f}%)")
print(f"  Cheap + Inefficient:   {cheap_ineff} ({cheap_ineff/len(cross)*100:.0f}%)")
print(f"  Expensive + Efficient: {exp_eff} ({exp_eff/len(cross)*100:.0f}%)")
print(f"  Expensive + Inefficient: {exp_ineff} ({exp_ineff/len(cross)*100:.0f}%)")

Correlation: cost residual vs efficiency score: ρ=-0.564 (p=3.91e-24)
  → Negative: hospitals that charge more tend to be less efficient



  Saved: 05_cost_deviation_vs_efficiency.png
Quadrant distribution:
  Cheap + Efficient:     69 (25%)
  Cheap + Inefficient:   10 (4%)
  Expensive + Efficient: 66 (24%)
  Expensive + Inefficient: 126 (46%)


## 6. Identifying Potentially Unnecessary Hospitalizations

Which admissions could plausibly have been handled as outpatient? Indicators:
1. **Diagnostic-only admissions** — hospitalized just for imaging/tests, no treatment
2. **Same-day discharge with low-cost procedures** — effectively outpatient but billed as inpatient
3. **Procedures available in SIA** — the same procedure code exists in ambulatory billing

In [14]:
# --- Flag 1: Diagnostic-only admissions ---
diag_only = kidney[kidney["proc_category"] == "DIAGNOSTIC"].copy()
diag_sameday = diag_only[diag_only["DIAS_PERM"] == 0]
diag_short = diag_only[diag_only["DIAS_PERM"] <= 1]

print("=== FLAG 1: DIAGNOSTIC-ONLY ADMISSIONS ===")
print(f"  Total diagnostic admissions: {len(diag_only):,}")
print(f"  Same-day discharge (D0):     {len(diag_sameday):,} ({len(diag_sameday)/len(diag_only)*100:.1f}%)")
print(f"  ≤1 day stay:                 {len(diag_short):,} ({len(diag_short)/len(diag_only)*100:.1f}%)")
print(f"  Total cost of diagnostic D0: R${diag_sameday['VAL_TOT'].sum():,.0f}")
print(f"  Total cost of diagnostic ≤1d:R${diag_short['VAL_TOT'].sum():,.0f}")

# --- Flag 2: Observation admissions (already short by nature) ---
obs_only = kidney[kidney["proc_category"] == "OBSERVATION"].copy()
print(f"\n=== FLAG 2: OBSERVATION ADMISSIONS ===")
print(f"  Total observation admissions: {len(obs_only):,}")
print(f"  Mean LOS: {obs_only['DIAS_PERM'].mean():.1f}d | Mean cost: R${obs_only['VAL_TOT'].mean():.0f}")
print(f"  Total cost: R${obs_only['VAL_TOT'].sum():,.0f}")

# --- Flag 3: Same-day discharge across ALL categories ---
sameday_all = kidney[kidney["DIAS_PERM"] == 0].copy()
print(f"\n=== FLAG 3: ALL SAME-DAY DISCHARGES (D0) ===")
print(f"  Total D0 admissions: {len(sameday_all):,} ({len(sameday_all)/len(kidney)*100:.1f}%)")
print(f"  Total cost: R${sameday_all['VAL_TOT'].sum():,.0f}")
d0_by_cat = sameday_all.groupby("proc_category").agg(
    n=("VAL_TOT", "count"),
    total_cost=("VAL_TOT", "sum"),
    mean_cost=("VAL_TOT", "mean"),
).sort_values("n", ascending=False)
print(f"\n  D0 breakdown by category:")
for cat, row in d0_by_cat.iterrows():
    pct_of_cat = row["n"] / len(kidney[kidney["proc_category"] == cat]) * 100
    print(f"    {cat:18s} | n={row['n']:>6,.0f} | R${row['total_cost']:>10,.0f} | mean=R${row['mean_cost']:>6,.0f} | {pct_of_cat:.0f}% of category")

# --- Flag 4: Procedures that exist in SIA (ambulatory) but were billed as inpatient ---
if not premium.empty:
    sia_procedures = set(premium.index)
    sia_inpatient = kidney[kidney["PROC_REA"].isin(sia_procedures)].copy()
    sia_d0 = sia_inpatient[sia_inpatient["DIAS_PERM"] == 0]
    sia_short = sia_inpatient[sia_inpatient["DIAS_PERM"] <= 1]
    
    print(f"\n=== FLAG 4: SIA-AVAILABLE PROCEDURES DONE AS INPATIENT ===")
    print(f"  Procedures found in both SIH and SIA: {len(sia_procedures)}")
    print(f"  Inpatient admissions for these procs: {len(sia_inpatient):,}")
    print(f"  Of those, same-day (D0): {len(sia_d0):,}")
    print(f"  Of those, ≤1 day: {len(sia_short):,}")
    print(f"  Total inpatient cost: R${sia_inpatient['VAL_TOT'].sum():,.0f}")
    
    # What would it cost at SIA rates?
    sia_rate_map = premium["sia_mean_cost"].to_dict()
    sia_inpatient["sia_cost"] = sia_inpatient["PROC_REA"].map(sia_rate_map)
    sia_inpatient["cost_diff"] = sia_inpatient["VAL_TOT"] - sia_inpatient["sia_cost"]
    
    total_sih = sia_inpatient["VAL_TOT"].sum()
    total_sia = sia_inpatient["sia_cost"].sum()
    print(f"\n  If ALL were billed at SIA rates:")
    print(f"    SIH total:    R${total_sih:>12,.0f}")
    print(f"    SIA total:    R${total_sia:>12,.0f}")
    print(f"    Difference:   R${total_sih - total_sia:>12,.0f} ({(total_sih-total_sia)/total_sih*100:.1f}%)")
    
    # More realistic: only D0 cases
    d0_sih = sia_d0["VAL_TOT"].sum()
    d0_sia = sia_d0["PROC_REA"].map(sia_rate_map).sum()
    print(f"\n  If only D0 cases were billed at SIA rates:")
    print(f"    SIH total:    R${d0_sih:>12,.0f}")
    print(f"    SIA total:    R${d0_sia:>12,.0f}")
    print(f"    Difference:   R${d0_sih - d0_sia:>12,.0f}")

=== FLAG 1: DIAGNOSTIC-ONLY ADMISSIONS ===
  Total diagnostic admissions: 41,487
  Same-day discharge (D0):     2,816 (6.8%)
  ≤1 day stay:                 14,695 (35.4%)
  Total cost of diagnostic D0: R$828,512
  Total cost of diagnostic ≤1d:R$4,847,826

=== FLAG 2: OBSERVATION ADMISSIONS ===
  Total observation admissions: 8,818
  Mean LOS: 0.6d | Mean cost: R$135
  Total cost: R$1,186,696

=== FLAG 3: ALL SAME-DAY DISCHARGES (D0) ===
  Total D0 admissions: 26,023 (12.6%)
  Total cost: R$18,003,290



  D0 breakdown by category:
    SURGICAL           | n=10,108 | R$ 7,730,214 | mean=R$   765 | 11% of category
    OBSERVATION        | n= 4,568 | R$   505,934 | mean=R$   111 | 52% of category
    CLINICAL_MGMT      | n= 3,259 | R$ 4,520,472 | mean=R$ 1,387 | 14% of category
    INTERVENTIONAL     | n= 3,016 | R$ 2,238,541 | mean=R$   742 | 15% of category
    DIAGNOSTIC         | n= 2,816 | R$   828,512 | mean=R$   294 | 7% of category
    SURGICAL_MGMT      | n= 1,740 | R$ 2,032,892 | mean=R$ 1,168 | 8% of category
    OTHER              | n=   516 | R$   146,724 | mean=R$   284 | 15% of category

=== FLAG 4: SIA-AVAILABLE PROCEDURES DONE AS INPATIENT ===
  Procedures found in both SIH and SIA: 13
  Inpatient admissions for these procs: 41,613
  Of those, same-day (D0): 3,984
  Of those, ≤1 day: 13,910
  Total inpatient cost: R$31,273,683

  If ALL were billed at SIA rates:
    SIH total:    R$  31,273,683
    SIA total:    R$   5,461,690
    Difference:   R$  25,811,993 (82.5%)

 

### 6b. Composite Unnecessary Hospitalization Score

Combine the flags into a per-admission score to quantify the likelihood of being unnecessary.

In [15]:
# Composite score: how many "unnecessary" flags does each admission have?
kidney["flag_diagnostic"] = (kidney["proc_category"] == "DIAGNOSTIC").astype(int)
kidney["flag_observation"] = (kidney["proc_category"] == "OBSERVATION").astype(int)
kidney["flag_d0"] = (kidney["DIAS_PERM"] == 0).astype(int)
kidney["flag_short"] = (kidney["DIAS_PERM"] <= 1).astype(int)
kidney["flag_sia_available"] = kidney["PROC_REA"].isin(sia_procedures if not premium.empty else set()).astype(int)
kidney["flag_low_cost"] = (kidney["VAL_TOT"] < kidney["VAL_TOT"].quantile(0.25)).astype(int)

kidney["unnecessary_score"] = (
    kidney["flag_diagnostic"] +
    kidney["flag_observation"] +
    kidney["flag_d0"] +
    kidney["flag_sia_available"] +
    kidney["flag_low_cost"]
)

print("=== UNNECESSARY HOSPITALIZATION SCORE DISTRIBUTION ===")
score_dist = kidney["unnecessary_score"].value_counts().sort_index()
for score, count in score_dist.items():
    pct = count / len(kidney) * 100
    cost = kidney[kidney["unnecessary_score"] == score]["VAL_TOT"].sum()
    print(f"  Score {score}: {count:>7,} admissions ({pct:5.1f}%) | R${cost:>12,.0f}")

# High-suspicion: score >= 3
high_sus = kidney[kidney["unnecessary_score"] >= 3]
print(f"\n=== HIGH SUSPICION (score ≥ 3) ===")
print(f"  Admissions: {len(high_sus):,} ({len(high_sus)/len(kidney)*100:.1f}%)")
print(f"  Total cost: R${high_sus['VAL_TOT'].sum():,.0f} ({high_sus['VAL_TOT'].sum()/kidney['VAL_TOT'].sum()*100:.1f}% of total)")
print(f"  Mean LOS: {high_sus['DIAS_PERM'].mean():.1f}d")
print(f"  Mortality: {high_sus['MORTE'].mean()*100:.2f}%")

# Which hospitals have the highest proportion of high-suspicion cases?
hosp_sus = kidney.groupby("CNES").agg(
    n=("CNES", "count"),
    high_sus_n=("unnecessary_score", lambda x: (x >= 3).sum()),
    high_sus_pct=("unnecessary_score", lambda x: (x >= 3).mean()),
    total_cost=("VAL_TOT", "sum"),
    high_sus_cost=("VAL_TOT", lambda x: x[kidney.loc[x.index, "unnecessary_score"] >= 3].sum()),
).reset_index()
hosp_sus = hosp_sus[hosp_sus["n"] >= MIN_N].copy()
hosp_sus["name"] = hosp_sus["CNES"].apply(lambda c: hospital_name(c, names_df))
hosp_sus = hosp_sus.sort_values("high_sus_pct", ascending=False)

print(f"\n=== TOP 15 HOSPITALS BY % HIGH-SUSPICION ADMISSIONS ===")
print(f"{'Hospital':45s} {'N':>6s} {'Suspect':>8s} {'%':>6s} {'Suspect Cost':>13s}")
print("-" * 85)
for _, r in hosp_sus.head(15).iterrows():
    print(f"{r['name'][:45]:45s} {r['n']:6.0f} {r['high_sus_n']:8.0f} {r['high_sus_pct']*100:5.1f}% R${r['high_sus_cost']:>11,.0f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

score_counts = kidney["unnecessary_score"].value_counts().sort_index()
axes[0].bar(score_counts.index, score_counts.values, color="#7C3AED", edgecolor="white")
axes[0].set_xlabel("Unnecessary Hospitalization Score")
axes[0].set_ylabel("Number of Admissions")
axes[0].set_title("Score Distribution (0=likely necessary, 5=likely unnecessary)")

# Top hospitals by suspicion
top_sus = hosp_sus.head(15)
labels = [n[:30] for n in top_sus["name"]]
axes[1].barh(range(len(labels)), top_sus["high_sus_pct"] * 100, color="#DC2626")
axes[1].set_yticks(range(len(labels)))
axes[1].set_yticklabels(labels, fontsize=8)
axes[1].set_xlabel("% Admissions with Score ≥ 3")
axes[1].set_title("Hospitals with Most Suspect Admissions")
axes[1].invert_yaxis()

fig.suptitle("Potentially Unnecessary Hospitalizations", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "unnecessary_hospitalizations", prefix="05")

=== UNNECESSARY HOSPITALIZATION SCORE DISTRIBUTION ===
  Score 0:  98,014 admissions ( 47.5%) | R$ 124,630,274
  Score 1:  56,966 admissions ( 27.6%) | R$  49,125,585


  Score 2:  42,284 admissions ( 20.5%) | R$  12,084,318
  Score 3:   9,236 admissions (  4.5%) | R$   1,982,979

=== HIGH SUSPICION (score ≥ 3) ===
  Admissions: 9,236 (4.5%)
  Total cost: R$1,982,979 (1.1% of total)
  Mean LOS: 0.0d
  Mortality: 0.06%



=== TOP 15 HOSPITALS BY % HIGH-SUSPICION ADMISSIONS ===


Hospital                                           N  Suspect      %  Suspect Cost
-------------------------------------------------------------------------------------
AME DR LUIZ ROBERTO BARRADAS BARATA SAO PAULO    987      678  68.7% R$    270,305
HOSPITAL UNIVERSITARIO DA UFSCAR                  59       23  39.0% R$      5,545
SANTA CASA DE MONTE ALTO                         393      132  33.6% R$     19,801
HOSPITAL SAO LUIZ                                137       43  31.4% R$      6,078
UNIDADE DE RETAGUARDA DE URGENCIA E DIAGNOSTI     48       12  25.0% R$      2,126
HOSPITAL BENEFICIENTE SANTO ANTONIO ORLANDIA     267       66  24.7% R$     17,447
HOSPITAL UNIVERSITARIO SAO FRANCISCO NA PROV    1975      456  23.1% R$     37,710
SANTA CASA DE VINHEDO                           1101      250  22.7% R$     50,493
HOSPITAL GERAL DE VILA PENTEADO DR JOSE PANGE    172       36  20.9% R$      1,724
HOSPITAL DAS CLINICAS HCFAMEMA                  1810      363  20.1% R$    113,835


  Saved: 05_unnecessary_hospitalizations.png


## Summary Metrics

In [16]:
metrics = {
    "total_system_cost_brl": round(float(total_cost), 2),
    "excess_cost_median_brl": round(float(total_excess_median), 2),
    "excess_cost_median_pct": round(float(excess_pct_median), 1),
    "excess_cost_p75_brl": round(float(total_excess_p75), 2),
    "excess_cost_p75_pct": round(float(excess_pct_p75), 1),
    "excess_bed_days_median": int(kidney_cost["excess_days_median"].sum()),
    "excess_bed_days_p75": int(kidney_cost["excess_days_p75"].sum()),
    "mean_cost_per_admission": round(float(kidney["VAL_TOT"].mean()), 2),
    "median_cost_per_admission": round(float(kidney["VAL_TOT"].median()), 2),
    "sia_procedures_compared": len(premium),
    "max_admission_premium": round(float(premium["admission_premium"].max()), 1) if not premium.empty else 0,
    "hospitals_analyzed": len(hosp_eff),
    "efficiency_range": f"{hosp_eff['efficiency'].min():.2f} to {hosp_eff['efficiency'].max():.2f}",
    "top15_avg_resolution": round(float(top15["w_resolution"].mean()), 3),
    "bot15_avg_resolution": round(float(bot15["w_resolution"].mean()), 3),
    "top15_avg_cost_ratio": round(float(top15["w_cost_ratio"].mean()), 3),
    "bot15_avg_cost_ratio": round(float(bot15["w_cost_ratio"].mean()), 3),
}
save_metrics(metrics, "05_financial_analysis")

print("\n=== FINANCIAL ANALYSIS ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

  Saved metrics: 05_financial_analysis.json

=== FINANCIAL ANALYSIS ===
  total_system_cost_brl: 187823155.55
  excess_cost_median_brl: 41771443.12
  excess_cost_median_pct: 22.2
  excess_cost_p75_brl: 27435391.36
  excess_cost_p75_pct: 14.6
  excess_bed_days_median: 212761
  excess_bed_days_p75: 149423
  mean_cost_per_admission: 909.56
  median_cost_per_admission: 766.11
  sia_procedures_compared: 13
  max_admission_premium: 1752.0
  hospitals_analyzed: 271
  efficiency_range: 0.10 to 1.47
  top15_avg_resolution: 0.856
  bot15_avg_resolution: 0.266
  top15_avg_cost_ratio: 0.706
  bot15_avg_cost_ratio: 1.48
